In [4]:
import pandas as pd
df = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
print(df.isnull().sum())
print("\nRows with missing values:\n")
print(df[df.isnull().any(axis=1)])

month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

Rows with missing values:

      month  sip_inflow_crore  active_sip_accounts_crore  \
0   2022-01             11517                       4.91   
1   2022-02             11438                       4.93   
2   2022-03             12328                       5.09   
3   2022-04             11863                       5.48   
4   2022-05             12286                       5.55   
5   2022-06             12276                       5.60   
6   2022-07             12140                       5.65   
7   2022-08             12694                       5.71   
8   2022-09             12976                       5.80   
9   2022-10             13040                       5.93   
10  2022-11             13306                       6.00   
11  2022-12             13573                  

In [9]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../data/db/bluestock_mf.db")

funds = pd.read_csv("../data/raw/01_fund_master.csv")
nav = pd.read_csv("../data/processed/clean_nav_history.csv")
transactions = pd.read_csv("../data/processed/clean_transactions.csv")
performance = pd.read_csv("../data/processed/clean_performance.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")

funds.to_sql("dim_fund", engine, if_exists="replace", index=False)
nav.to_sql("fact_nav", engine, if_exists="replace", index=False)
transactions.to_sql("fact_transactions", engine, if_exists="replace", index=False)
performance.to_sql("fact_performance", engine, if_exists="replace", index=False)
aum.to_sql("fact_aum", engine, if_exists="replace", index=False)

print("Database Loaded Successfully!")

Database Loaded Successfully!


In [11]:
import sqlite3

conn = sqlite3.connect("../data/db/bluestock_mf.db")
tables = ["dim_fund", "fact_nav", "fact_transactions", "fact_performance", "fact_aum"]
for table in tables:
    query = f"""SELECT COUNT(*) AS rows FROM {table}"""
    result = pd.read_sql(query, conn)
    print("\n", table)
    print(result)
conn.close()


 dim_fund
   rows
0    40

 fact_nav
    rows
0  46000

 fact_transactions
    rows
0  32778

 fact_performance
   rows
0    40

 fact_aum
   rows
0    90


In [12]:
fund_codes = set(funds["amfi_code"])
nav_codes = set(nav["amfi_code"])

missing = fund_codes - nav_codes

print("Missing codes:", len(missing))
print(missing)

Missing codes: 0
set()
